In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

SIGNALS_PATH = '../data/processed/signals.csv'

## 1. Load & basic shape

In [ ]:
df = pd.read_csv(SIGNALS_PATH, parse_dates=['earnings_date', 'reaction_date'])
df['reaction_date'] = pd.to_datetime(df['reaction_date'])
df['year']    = df['reaction_date'].dt.year
df['quarter'] = df['reaction_date'].dt.to_period('Q')

print(f"Rows: {len(df):,}")
print(f"Tickers: {df['ticker'].nunique():,}")
print(f"Date range: {df['reaction_date'].min().date()} → {df['reaction_date'].max().date()}")
print()
print(df[['gap','prev_close','open_t','eps_estimate','reported_eps','surprise_pct']].describe().round(4))

## 2. Session split

In [ ]:
session_counts = df['session'].value_counts()
print(session_counts)

fig, ax = plt.subplots(figsize=(5, 3))
session_counts.plot(kind='bar', ax=ax, color=['#2196F3','#FF9800'], edgecolor='white')
ax.set_xlabel('')
ax.set_ylabel('Events')
ax.set_title('BMO vs AMC event count')
ax.bar_label(ax.containers[0], fmt='{:,.0f}')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 3. Gap distribution — full universe

In [ ]:
gap_pct = df['gap'] * 100  # convert to %

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(gap_pct.clip(-20, 20), bins=120, color='#2196F3', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--', label='zero')
ax.axvline(gap_pct.mean(), color='red', linewidth=1.2, linestyle='-', label=f'mean {gap_pct.mean():.2f}%')
ax.set_xlabel('Overnight gap (%)')
ax.set_ylabel('Events')
ax.set_title('Gap distribution (clipped at ±20% for readability)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Events outside ±20%: {(gap_pct.abs() > 20).sum()} ({(gap_pct.abs() > 20).mean()*100:.1f}%)")

## 4. BMO vs AMC gap distributions

In [ ]:
bmo = df[df['session'] == 'bmo']['gap'] * 100
amc = df[df['session'] == 'amc']['gap'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, series, label, color in zip(axes, [bmo, amc], ['BMO', 'AMC'], ['#2196F3', '#FF9800']):
    ax.hist(series.clip(-20, 20), bins=80, color=color, edgecolor='white', linewidth=0.3, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(series.mean(), color='red', linewidth=1.2,
               label=f'mean {series.mean():.2f}%  std {series.std():.2f}%')
    ax.set_title(f'{label}  (n={len(series):,})')
    ax.set_xlabel('Gap (%)')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Events')
fig.suptitle('BMO vs AMC gap distributions (clipped at ±20%)', y=1.01)
plt.tight_layout()
plt.show()

## 5. Mean gap by year

In [ ]:
annual = (
    df.groupby(['year', 'session'])['gap']
    .agg(mean_gap='mean', count='count')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 4))
for session, color in [('bmo', '#2196F3'), ('amc', '#FF9800')]:
    sub = annual[annual['session'] == session]
    ax.plot(sub['year'], sub['mean_gap'] * 100, marker='o', label=session.upper(),
            color=color, linewidth=1.5)

ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax.set_xlabel('Year')
ax.set_ylabel('Mean overnight gap (%)')
ax.set_title('Mean earnings gap by year — BMO vs AMC')
ax.legend()
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.show()

## 6. Event count by quarter (data completeness check)

In [ ]:
quarterly_counts = df.groupby('quarter').size()

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(len(quarterly_counts)), quarterly_counts.values, color='#607D8B', edgecolor='white', linewidth=0.2)
ax.axhline(quarterly_counts.mean(), color='red', linewidth=1, linestyle='--',
           label=f'mean {quarterly_counts.mean():.0f}/quarter')
ax.set_xticks(range(0, len(quarterly_counts), 4))
ax.set_xticklabels(
    [str(q) for q in quarterly_counts.index[::4]],
    rotation=45, ha='right', fontsize=8
)
ax.set_ylabel('Events')
ax.set_title('Earnings events per quarter (sanity check: expect ~500)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Extreme gaps — spot-check table

In [ ]:
cols = ['ticker', 'reaction_date', 'session', 'prev_close', 'open_t', 'gap',
        'reported_eps', 'surprise_pct']

print("── Top 15 positive gaps ──")
display(df.nlargest(15, 'gap')[cols].assign(gap=lambda x: (x['gap']*100).round(1)).reset_index(drop=True))

print("\n── Top 15 negative gaps ──")
display(df.nsmallest(15, 'gap')[cols].assign(gap=lambda x: (x['gap']*100).round(1)).reset_index(drop=True))

## 8. Flat threshold sweep

In [ ]:
thresholds = np.arange(0.5, 10.5, 0.5)
records = []
for t in thresholds:
    passing = df[df['gap'] > t / 100]
    records.append({
        'threshold_pct': t,
        'n_events': len(passing),
        'pct_passing': len(passing) / len(df) * 100,
        'mean_gap_pct': passing['gap'].mean() * 100 if len(passing) else 0,
    })

sweep = pd.DataFrame(records)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(sweep['threshold_pct'], sweep['pct_passing'], marker='o', color='#2196F3')
ax1.set_xlabel('Gap threshold (%)')
ax1.set_ylabel('% events passing')
ax1.set_title('Events passing flat threshold')
ax1.axvline(2.0, color='red', linestyle='--', linewidth=0.8, label='2% default')
ax1.legend()

ax2.plot(sweep['threshold_pct'], sweep['mean_gap_pct'], marker='o', color='#FF9800')
ax2.set_xlabel('Gap threshold (%)')
ax2.set_ylabel('Mean gap of passing events (%)')
ax2.set_title('Mean gap above threshold')
ax2.axvline(2.0, color='red', linestyle='--', linewidth=0.8, label='2% default')
ax2.legend()

plt.tight_layout()
plt.show()

print(sweep.to_string(index=False))

## 9. Friday AMC events

In [ ]:
friday_amc = df[df['friday_amc'] == True]
non_friday_amc = df[(df['session'] == 'amc') & (df['friday_amc'] == False)]

print(f"Friday AMC events: {len(friday_amc):,} ({len(friday_amc)/len(df)*100:.1f}% of all events)")
print(f"Non-Friday AMC events: {len(non_friday_amc):,}")
print()
print(f"Friday AMC mean gap:     {friday_amc['gap'].mean()*100:.2f}%")
print(f"Non-Friday AMC mean gap: {non_friday_amc['gap'].mean()*100:.2f}%")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist((friday_amc['gap']*100).clip(-20,20), bins=50, alpha=0.7,
        label=f'Friday AMC (n={len(friday_amc):,})', color='#E91E63', edgecolor='white')
ax.hist((non_friday_amc['gap']*100).clip(-20,20), bins=50, alpha=0.5,
        label=f'Non-Friday AMC (n={len(non_friday_amc):,})', color='#FF9800', edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Gap (%)')
ax.set_ylabel('Events')
ax.set_title('Friday AMC vs non-Friday AMC gap distributions')
ax.legend()
plt.tight_layout()
plt.show()